# PyTorch 数据加载实验课 — Dataset & DataLoader

## 为什么需要 Dataset / DataLoader？

你的数据集可能是 **10 万张图片**、**1 亿行文本**。它们不可能一次性全部塞进 GPU 显存（一张 24GB 的卡塞不下 100GB 的图片）。

PyTorch 的解决方案是把「**怎么取一个样本**」和「**怎么批量喂给模型**」分开：

```
Dataset       → 定义「怎么读第 idx 个样本」     （你写的逻辑）
DataLoader    → 负责「批量打包 + 打乱 + 多进程预取」   （PyTorch 帮你做）
```

```
                       Dataset（你的数据）
                              │
                       DataLoader（打包 / 打乱 / 预取）
                              │
                         for x, y in loader:        ← 训练循环里只要写这一行
                              │
                          model(x) → loss → step
```

配合 `tutorial/05_data_loading.py` 学习。练习在 `practice/05_custom_dataset.py`。

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print("用到的模块: torch.utils.data (Dataset, DataLoader, TensorDataset, random_split)")

PyTorch version: 2.12.1
用到的模块: torch.utils.data (Dataset, DataLoader, TensorDataset, random_split)


## 1. Dataset 的两个魔法方法 — `__len__` 和 `__getitem__`

一个 `Dataset` 子类**只需要实现两个方法**，DataLoader 就能用它：

| 方法 | 返回 | 作用 |
|------|------|------|
| `__len__(self)` | `int` | 数据集多大？（有多少个样本） |
| `__getitem__(self, idx)` | `(x, y)` 或 `x` | 第 `idx` 个样本长什么样？ |

为什么是这两个？因为 Python 的 `len(dataset)` 和 `dataset[i]` 语法糖分别调用它们。实现完，你的数据集就能像 list 一样用：

```python
ds = MyDataset(...)
print(len(ds))   # → 调用 ds.__len__()
x, y = ds[0]     # → 调用 ds.__getitem__(0)
```

> 💡 **`__getitem__` 是 DataLoader 多进程调用的入口**，所以它必须**线程安全**（不要在里面写共享全局变量）。读文件、做 transform 都没问题。

In [2]:
# --- 最简单的自定义 Dataset：数数 ---
print("=" * 60)
print("自定义 Dataset — 只需实现 __len__ + __getitem__")
print("=" * 60)


class RangeDataset(Dataset):
    """生成 [0, n-1] 的整数 Dataset，每个样本返回 torch.long tensor。"""

    def __init__(self, n: int):
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return torch.tensor(idx, dtype=torch.long)


ds = RangeDataset(5)

print(f"len(ds) = {len(ds)}")        # → __len__()
print(f"ds[0]   = {ds[0]}")          # → __getitem__(0)
print(f"ds[3]   = {ds[3]}")
print(f"ds[-1]  = {ds[-1]}  ← 负索引也行（和 list 一样）")
print()
print("👉 只写了两个方法，Dataset 就完整了。" )

自定义 Dataset — 只需实现 __len__ + __getitem__
len(ds) = 5
ds[0]   = 0
ds[3]   = 3
ds[-1]  = -1  ← 负索引也行（和 list 一样）

👉 只写了两个方法，Dataset 就完整了。


## 2. TensorDataset — 数据已在内存里时的偷懒神器

如果你已经有两个对齐好的 tensor：特征 `X` 和标签 `y`（第一维大小必须相同），不用自己写 `Dataset` 子类，直接用 `TensorDataset`：

```python
dataset = TensorDataset(X, y)     # X.shape[0] 必须等于 y.shape[0]
dataset[0]                        # → (X[0], y[0])，一个 tuple
```

In [3]:
# --- TensorDataset：把已有 tensor 包装成 Dataset ---
print("=" * 60)
print("TensorDataset — 零样板代码的 Dataset")
print("=" * 60)

X = torch.randn(100, 5)            # 100 个样本，每个 5 维
y = torch.randint(0, 3, (100,))    # 100 个标签，3 分类（值 0/1/2）

dataset = TensorDataset(X, y)

print(f"len(dataset) = {len(dataset)}")
print(f"dataset[0] 返回的是个 tuple，长度 = {len(dataset[0])}")
print(f"  feature: {dataset[0][0][:3]}...   shape={tuple(dataset[0][0].shape)}")
print(f"  label:   {dataset[0][1]}                    dtype={dataset[0][1].dtype}")
print()

# TensorDataset 实现的就是这么简单的东西：
print("它内部等价于这个手写 Dataset:")
print("""
class TensorDataset(Dataset):
    def __init__(self, *tensors):
        assert all(t.shape[0] == tensors[0].shape[0] for t in tensors)
        self.tensors = tensors
    def __len__(self):
        return self.tensors[0].shape[0]
    def __getitem__(self, idx):
        return tuple(t[idx] for t in self.tensors)
""")

TensorDataset — 零样板代码的 Dataset
len(dataset) = 100
dataset[0] 返回的是个 tuple，长度 = 2
  feature: tensor([ 0.6840, -0.1744,  1.3571])...   shape=(5,)
  label:   0                    dtype=torch.int64

它内部等价于这个手写 Dataset:

class TensorDataset(Dataset):
    def __init__(self, *tensors):
        assert all(t.shape[0] == tensors[0].shape[0] for t in tensors)
        self.tensors = tensors
    def __len__(self):
        return self.tensors[0].shape[0]
    def __getitem__(self, idx):
        return tuple(t[idx] for t in self.tensors)



## 3. 自定义 Dataset 模板 — 带数据预处理

真实场景里你的数据多半是 numpy array、文件路径列表、或大到放不进内存的文件。`__getitem__` 里要做两件事：

1. **取出**第 `idx` 个样本（从内存、磁盘、数据库）
2. **预处理**它（转 tensor、标准化、数据增强…）

把预处理写在 `__getitem__` 而不是 `__init__` 里有个大好处：**数据可以懒加载**——只有真正被取到的样本才会被读进内存。这就是为什么 PyTorch 能训练放不下内存的数据集。

In [4]:
# --- 完整的自定义 Dataset 模板 ---
print("=" * 60)
print("自定义 Dataset 模板 — numpy 数据 + transform")
print("=" * 60)


class LabeledDataset(Dataset):
    """通用模板：numpy 数据 + 可选 transform。"""

    def __init__(self, data, labels, transform=None):
        self.data = data
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]

        # 关键：把 numpy 转成 torch，并指定 dtype
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.long)

        if self.transform is not None:
            x = self.transform(x)

        return x, y


# 造数据
raw_data = np.random.randn(100, 5).astype(np.float32)
raw_labels = np.random.randint(0, 3, 100)

# 不带 transform
ds_plain = LabeledDataset(raw_data, raw_labels)
x0, y0 = ds_plain[0]
print(f"不带 transform: x.dtype={x0.dtype}, y.dtype={y0.dtype}, x={x0[:3].tolist()}...")

# 带 transform：每个特征 +10
ds_tf = LabeledDataset(raw_data, raw_labels, transform=lambda x: x + 10)
x1, y1 = ds_tf[0]
print(f"带 transform(+10): x={x1[:3].tolist()}...")
print(f"  → 原始 {x0[:3].tolist()}  +10  → {x1[:3].tolist()}  ✅")

自定义 Dataset 模板 — numpy 数据 + transform
不带 transform: x.dtype=torch.float32, y.dtype=torch.int64, x=[-0.5065618753433228, 0.29887983202934265, 1.0474082231521606]...
带 transform(+10): x=[9.493437767028809, 10.298879623413086, 11.047408103942871]...
  → 原始 [-0.5065618753433228, 0.29887983202934265, 1.0474082231521606]  +10  → [9.493437767028809, 10.298879623413086, 11.047408103942871]  ✅


## 4. DataLoader 核心参数 — 批处理、打乱、多进程

`Dataset` 只回答「**第 i 个样本是谁**」，`DataLoader` 负责「**怎么把样本组织成 batch 喂给模型**」。

```python
loader = DataLoader(
    dataset,
    batch_size=16,        # 每批多少样本
    shuffle=True,         # 每个 epoch 是否重新打乱顺序
    num_workers=0,        # 用几个子进程并行加载（0 = 主进程）
    drop_last=False,      # 最后不足一个 batch 的要不要丢掉
    pin_memory=False,     # 锁页内存，加速 CPU→GPU 拷贝（有 GPU 时开 True）
)
```

`for x, y in loader:` 会自动遍历完整个数据集。`len(loader)` = 总 batch 数。

In [5]:
# --- DataLoader：把 Dataset 切成 batch ---
print("=" * 60)
print("DataLoader — batch_size / drop_last")
print("=" * 60)

dataset = TensorDataset(torch.randn(100, 5), torch.randint(0, 3, (100,)))

# 默认：drop_last=False
loader = DataLoader(dataset, batch_size=16, shuffle=False, drop_last=False)
print(f"batch_size=16, 共 {len(dataset)} 个样本, drop_last=False")
print(f"→ {len(loader)} 个 batch (因为 100/16 = 6 余 4，不丢尾就是 7 个)\n")

batch_sizes = []
for batch_idx, (x, y) in enumerate(loader):
    batch_sizes.append(x.shape[0])
    print(f"  Batch {batch_idx}: x.shape={tuple(x.shape)}, y.shape={tuple(y.shape)}")

print(f"\n  各 batch 大小: {batch_sizes}")
print(f"  最后一个 batch 只有 {batch_sizes[-1]} 个样本（不足 16）")

# drop_last=True：丢弃不完整的最后一个 batch
loader_drop = DataLoader(dataset, batch_size=16, drop_last=True)
print(f"\ndrop_last=True: {len(loader_drop)} 个 batch（最后 4 个样本被丢弃）")
print("  ↑ 训练时常用，尤其网络里有 BatchNorm 时，batch 太小统计会不稳")

DataLoader — batch_size / drop_last
batch_size=16, 共 100 个样本, drop_last=False
→ 7 个 batch (因为 100/16 = 6 余 4，不丢尾就是 7 个)

  Batch 0: x.shape=(16, 5), y.shape=(16,)
  Batch 1: x.shape=(16, 5), y.shape=(16,)
  Batch 2: x.shape=(16, 5), y.shape=(16,)
  Batch 3: x.shape=(16, 5), y.shape=(16,)
  Batch 4: x.shape=(16, 5), y.shape=(16,)
  Batch 5: x.shape=(16, 5), y.shape=(16,)
  Batch 6: x.shape=(4, 5), y.shape=(4,)

  各 batch 大小: [16, 16, 16, 16, 16, 16, 4]
  最后一个 batch 只有 4 个样本（不足 16）

drop_last=True: 6 个 batch（最后 4 个样本被丢弃）
  ↑ 训练时常用，尤其网络里有 BatchNorm 时，batch 太小统计会不稳


## 5. shuffle — 训练时开，验证时关

`shuffle=True` 会在**每个 epoch 开始时**重新随机打乱数据顺序。

| 场景 | shuffle | 为什么 |
|------|---------|--------|
| **训练集** | `True` | 防止模型记住数据顺序；让每个 batch 的样本分布更随机 |
| **验证/测试集** | `False` | 结果要可复现；顺序不影响评估指标 |

不开 shuffle 会怎样？如果数据是按类别排好序的（先 1000 个猫、再 1000 个狗…），一个 batch 里全是同一类，梯度方向会非常极端，训练崩溃。

In [6]:
# --- shuffle 的效果 ---
print("=" * 60)
print("shuffle — 看顺序怎么变")
print("=" * 60)

# 用 0..9 作为样本，方便看清顺序
dataset = TensorDataset(torch.arange(10), torch.arange(10) * 10)

print("shuffle=False（顺序不变）:")
loader_ordered = DataLoader(dataset, batch_size=4, shuffle=False)
for i, (x, _) in enumerate(loader_ordered):
    print(f"  Batch {i}: {x.tolist()}")

print("\nshuffle=True（每个 epoch 顺序随机）:")
torch.manual_seed(0)
loader_shuffled = DataLoader(dataset, batch_size=4, shuffle=True)
for epoch in range(2):
    batches = [x.tolist() for x, _ in loader_shuffled]
    print(f"  Epoch {epoch}: {batches}")

print("\n→ 同一个 loader，两个 epoch 的顺序不一样（每次重新打乱）")

shuffle — 看顺序怎么变
shuffle=False（顺序不变）:
  Batch 0: [0, 1, 2, 3]
  Batch 1: [4, 5, 6, 7]
  Batch 2: [8, 9]

shuffle=True（每个 epoch 顺序随机）:
  Epoch 0: [[6, 7, 1, 4], [2, 0, 9, 8], [3, 5]]
  Epoch 1: [[2, 4, 7, 0], [8, 9, 5, 3], [6, 1]]

→ 同一个 loader，两个 epoch 的顺序不一样（每次重新打乱）


## 6. collate_fn — 怎么把多个样本拼成一个 batch

DataLoader 取数据的过程是：

```
dataset[0] → (x0, y0)     ┐
dataset[1] → (x1, y1)     ├→  collate_fn(样本列表)  →  (batch_x, batch_y)
dataset[2] → (x2, y2)     ┘          ↑
                            默认用 torch.stack 把同位置的张量堆起来
```

默认的 collate 行为对 99% 的场景够用。但当你遇到**变长序列**（句子、时间序列）或想**自定义 batch 结构**时，就得写自己的 `collate_fn`。

In [7]:
# --- collate_fn：手动堆叠 batch ---
print("=" * 60)
print("collate_fn — 理解默认行为，再写自定义版本")
print("=" * 60)


# 默认 collate 等价于这样：
def my_collate_fn(batch):
    """
    batch 是一个 list，每个元素是 __getitem__ 的返回值。
    这里：每个元素是 (x_tensor, y_tensor)。
    """
    # 把所有 x 堆成一个张量，所有 y 堆成一个张量
    xs = [item[0] for item in batch]
    ys = [item[1] for item in batch]
    return torch.stack(xs), torch.stack(ys)


dataset = TensorDataset(torch.tensor([[1., 2.], [3., 4.], [5., 6.]]),
                        torch.tensor([0, 1, 0]))

loader_default = DataLoader(dataset, batch_size=2, shuffle=False)
loader_custom = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=my_collate_fn)

bx_def, by_def = next(iter(loader_default))
bx_cus, by_cus = next(iter(loader_custom))

print(f"默认 collate:  x={bx_def.tolist()}, y={by_def.tolist()}")
print(f"自定义 collate: x={bx_cus.tolist()}, y={by_cus.tolist()}")
print(f"\n两者一致? {torch.allclose(bx_def, bx_cus) and torch.allclose(by_def, by_cus)} ✅")
print("\n→ 自定义 collate_fn 在变长文本（要 pad）、多模态输入时很有用")

collate_fn — 理解默认行为，再写自定义版本
默认 collate:  x=[[1.0, 2.0], [3.0, 4.0]], y=[0, 1]
自定义 collate: x=[[1.0, 2.0], [3.0, 4.0]], y=[0, 1]

两者一致? True ✅

→ 自定义 collate_fn 在变长文本（要 pad）、多模态输入时很有用


## 7. 数据预处理 — transforms

`transform` 就是个**普通函数**：输入样本，输出处理后的样本。`torchvision.transforms` 提供了一批现成的（针对图片）：

| Transform | 作用 |
|-----------|------|
| `T.ToTensor()` | PIL/numpy → Tensor，并把图片像素从 `[0,255]` 归一化到 `[0,1]` |
| `T.Resize((224,224))` | 统一尺寸 |
| `T.Normalize(mean, std)` | 标准化：`(x - mean) / std` |
| `T.RandomHorizontalFlip()` | **数据增强**：随机水平翻转 |
| `T.Compose([...])` | 把多个 transform 串成流水线 |

对非图片数据（表格、特征向量），直接写 lambda 或普通函数就行。

In [8]:
# --- transforms：自己写 + torchvision ---
print("=" * 60)
print("transforms — 自定义函数 vs torchvision")
print("=" * 60)

# ① 自定义 transform：标准化（适用于任意 tensor）
def create_normalize_transform(mean, std):
    return lambda x: (x - mean) / std

x = torch.tensor([3.0, 10.0])
mean = torch.tensor([1.0, 2.0])
std = torch.tensor([2.0, 4.0])
normalize = create_normalize_transform(mean, std)
out = normalize(x)
print(f"自定义标准化: ({x.tolist()} - {mean.tolist()}) / {std.tolist()} = {out.tolist()}")
print(f"  → 验证: (3-1)/2=1.0, (10-2)/4=2.0  ✅")

# ② torchvision transforms（针对图片）
print("\ntorchvision 标准预处理流水线（ImageNet 预训练模型用）:")
try:
    import torchvision.transforms as T

    transform = T.Compose([
        T.Resize((224, 224)),           # 统一尺寸
        T.RandomHorizontalFlip(),       # 数据增强：随机翻转
        T.ToTensor(),                   # → Tensor，像素归一化到 [0,1]
        T.Normalize(                    # 用 ImageNet 的均值/方差标准化
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])
    print("  Resize(224) → RandomHorizontalFlip → ToTensor → Normalize")
    print("  ↑ Compose 把它们串起来，一行 transform(img) 跑完整个流水线")
except ImportError:
    print("  (torchvision 未安装)")

transforms — 自定义函数 vs torchvision
自定义标准化: ([3.0, 10.0] - [1.0, 2.0]) / [2.0, 4.0] = [1.0, 2.0]
  → 验证: (3-1)/2=1.0, (10-2)/4=2.0  ✅

torchvision 标准预处理流水线（ImageNet 预训练模型用）:


  Resize(224) → RandomHorizontalFlip → ToTensor → Normalize
  ↑ Compose 把它们串起来，一行 transform(img) 跑完整个流水线


## 8. 数据集切分 — train / val / test

训练集用来学，验证集用来调超参 / 早停，测试集只在最后评估一次。三者**绝不能混用**。

PyTorch 提供了 `random_split` 做随机切分；也可以手动用 `torch.randperm` 生成打乱的索引再切（更可控）。

In [9]:
# --- 切分 train / val ---
print("=" * 60)
print("数据集切分 — random_split vs 手动 randperm")
print("=" * 60)

dataset = TensorDataset(torch.arange(10).float().unsqueeze(1), torch.arange(10))

# 方法 A：random_split（最省事）
train_ds_a, val_ds_a = random_split(dataset, [6, 4], generator=torch.Generator().manual_seed(0))
print(f"random_split([6,4]): train={len(train_ds_a)}, val={len(val_ds_a)}")

# 方法 B：手动 randperm（完全可控，切完还是 TensorDataset）
def split_dataset(dataset, train_ratio):
    n = len(dataset)
    perm = torch.randperm(n)                          # 0..n-1 的随机排列
    n_train = int(n * train_ratio)
    train_idx, val_idx = perm[:n_train], perm[n_train:]

    # 取出打乱后的所有样本（这里 dataset 是 TensorDataset，可以整体索引）
    all_x = dataset.tensors[0][train_idx]
    all_y = dataset.tensors[1][train_idx]
    val_x = dataset.tensors[0][val_idx]
    val_y = dataset.tensors[1][val_idx]
    return TensorDataset(all_x, all_y), TensorDataset(val_x, val_y)

torch.manual_seed(0)
train_ds_b, val_ds_b = split_dataset(dataset, 0.6)
print(f"手动 randperm(0.6): train={len(train_ds_b)}, val={len(val_ds_b)}")
print(f"  train 样本: {[int(y) for _, y in train_ds_b]}")
print(f"  val   样本: {[int(y) for _, y in val_ds_b]}")
print("\n→ 切分后通常再各包一个 DataLoader: train_loader / val_loader")

数据集切分 — random_split vs 手动 randperm
random_split([6,4]): train=6, val=4
手动 randperm(0.6): train=6, val=4
  train 样本: [4, 1, 7, 5, 3, 9]
  val   样本: [0, 8, 6, 2]

→ 切分后通常再各包一个 DataLoader: train_loader / val_loader


## 9. 真实场景：图片、文本、大数据集

`__getitem__` 的强大之处在于——它可以是**任何逻辑**。下面是几种典型场景的写法。

### 场景 A：图片在文件系统里

```python
class ImageDataset(Dataset):
    def __init__(self, file_list, labels, transform=None):
        self.file_list = file_list     # ['cat/001.jpg', 'cat/002.jpg', ...]
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        from PIL import Image
        img = Image.open(self.file_list[idx]).convert('RGB')  # ← 按需读磁盘
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]
```

关键点：**图片只在被取到时才读进内存**。100GB 的图片集也跑得动，因为同一时刻只有几个 batch 在内存里。

### 场景 B：torchvision 内置数据集（一行下载）

```python
from torchvision import datasets

# 自动下载 + 缓存到 ./data
cifar = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# 按文件夹自动分类（data/dog/*.jpg → label=dog, data/cat/*.jpg → label=cat）
folder = datasets.ImageFolder(root='./images/', transform=transform)
```

### 场景 C：文本 / HuggingFace

```python
from datasets import load_dataset
raw = load_dataset('imdb', split='train')   # HuggingFace datasets 库
```

In [10]:
# --- 场景演示：模拟一个「文件路径」Dataset ---
print("=" * 60)
print("模拟文件式 Dataset（不真的读文件，演示结构）")
print("=" * 60)


class FakeImageDataset(Dataset):
    """假装 file_list 是图片路径，实际用随机张量代替真实图片。"""

    def __init__(self, file_list, labels, transform=None):
        self.file_list = file_list
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        # 真实代码里是: img = Image.open(self.file_list[idx])
        # 这里假装读出来一张 3x4x4 的图
        img = torch.randn(3, 4, 4)
        if self.transform is not None:
            img = self.transform(img)
        return img, self.labels[idx]


files = [f'images/{i}.jpg' for i in range(50)]
labels = torch.randint(0, 2, (50,))  # 二分类

img_ds = FakeImageDataset(files, labels)
loader = DataLoader(img_ds, batch_size=8, shuffle=True)

imgs, lbls = next(iter(loader))
print(f"len(dataset)    = {len(img_ds)}")
print(f"一个 batch: imgs.shape={tuple(imgs.shape)}, labels.shape={tuple(lbls.shape)}")
print(f"                ↑ (batch=8, channels=3, H=4, W=4)")
print("\n→ 真实图片训练也就是这样，只是 __getitem__ 里换成 Image.open")

模拟文件式 Dataset（不真的读文件，演示结构）
len(dataset)    = 50
一个 batch: imgs.shape=(8, 3, 4, 4), labels.shape=(8,)
                ↑ (batch=8, channels=3, H=4, W=4)

→ 真实图片训练也就是这样，只是 __getitem__ 里换成 Image.open


## 10. 完整训练循环 — DataLoader + 模型 + 优化器

把前面所有东西串起来：用 `TensorDataset` 造数据 → 包成 `DataLoader` → 一个 batch 一个 batch 地训练。

注意一个常见 gotcha：**`len(loader)` 是 batch 数，不是样本数**。想算总样本数要用 `len(dataset)`。

In [11]:
# --- 完整训练循环 ---
print("=" * 60)
print("完整训练循环 — DataLoader 喂数据")
print("=" * 60)

import torch.nn as nn
import torch.optim as optim

# ① 造数据：3 分类，5 维特征
torch.manual_seed(42)
X = torch.randn(200, 5)
y = (X[:, 0] + X[:, 1] > 0).long()   # 简单规律：前两维之和 > 0 → 类 1

dataset = TensorDataset(X, y)
train_ds, val_ds = random_split(dataset, [160, 40], generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

print(f"总样本: {len(dataset)}, train: {len(train_ds)}, val: {len(val_ds)}")
print(f"train_loader: {len(train_loader)} 个 batch (160/32=5)")
print(f"val_loader:   {len(val_loader)} 个 batch (40/32=2)\n")

# ② 模型 / 损失 / 优化器
model = nn.Linear(5, 2)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# ③ 训练 10 个 epoch
for epoch in range(10):
    # ---- 训练阶段 ----
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        pred = model(x_batch)
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(x_batch)
    train_loss /= len(train_ds)

    # ---- 验证阶段 ----
    model.eval()
    correct = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            pred = model(x_batch)
            correct += (pred.argmax(dim=1) == y_batch).sum().item()
    val_acc = correct / len(val_ds)

    if epoch % 2 == 0 or epoch == 9:
        print(f"Epoch {epoch:2d}  train_loss={train_loss:.4f}  val_acc={val_acc:.2f}")

print("\n✅ 数据→Dataset→DataLoader→训练循环，全链路跑通了")

完整训练循环 — DataLoader 喂数据
总样本: 200, train: 160, val: 40
train_loader: 5 个 batch (160/32=5)
val_loader:   2 个 batch (40/32=2)

Epoch  0  train_loss=0.8476  val_acc=0.55
Epoch  2  train_loss=0.4727  val_acc=0.95
Epoch  4  train_loss=0.3492  val_acc=0.97
Epoch  6  train_loss=0.2928  val_acc=0.97
Epoch  8  train_loss=0.2587  val_acc=0.97
Epoch  9  train_loss=0.2463  val_acc=0.97

✅ 数据→Dataset→DataLoader→训练循环，全链路跑通了


## 11. 性能参数 — `num_workers` 和 `pin_memory`

数据加载往往是训练的瓶颈（GPU 算得飞快，却在等 CPU 读图）。两个参数能大幅加速：

### `num_workers`（多进程加载）

```
num_workers=0   → 主进程读数据（GPU 算完 batch 就得干等 CPU）
num_workers=4   → 开 4 个子进程并行预取下一个 batch（GPU 几乎不等）
```

- 像 ImageNet 这种 `__getitem__` 里要解码 JPEG 的场景，效果立竿见影
- ⚠️ **Windows / Jupyter 里有时会报错**（多进程的 pickle 问题），调试时先设 0，跑通后再调大

### `pin_memory`（锁页内存）

```
普通内存  ──CPU→GPU拷贝──→  GPU
锁页内存  ──异步直达────→  GPU   ← 快！
```

- `pin_memory=True` 把 batch 放进「锁页内存」，CPU→GPU 拷贝能用异步 DMA，更快
- 配合训练循环里 `x = x.to(device, non_blocking=True)` 效果最好
- 没 GPU 时没用，有 GPU 时建议开

> 经验法则：**小数据 / 调试** → `num_workers=0`；**真实训练（图片、大文本）** → `num_workers=4~8, pin_memory=True`。

In [12]:
# --- num_workers：概念演示（不开真正的子进程）---
print("=" * 60)
print("num_workers — 串行 vs 并行加载的概念演示")
print("=" * 60)

import time

# 模拟「__getitem__ 干重活」的串行耗时：只测 num_workers=0（永远安全）
class SlowDataset(Dataset):
    def __init__(self, n, work_ms=2.0):
        self.n, self.work = n, work_ms / 1000
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        time.sleep(self.work)        # 模拟「读图 + JPEG 解码」
        return torch.randn(3, 32, 32), idx % 10

n_samples, batch_size = 256, 32
slow_ds = SlowDataset(n_samples)
loader = DataLoader(slow_ds, batch_size=batch_size, shuffle=False, num_workers=0)

# warm + time
for x, y in loader:
    pass
t0 = time.perf_counter()
for x, y in loader:
    pass
serial_ms = (time.perf_counter() - t0) * 1000

print(f"数据集 {n_samples} 样本, batch_size={batch_size}, 每个 __getitem__ 模拟 2ms")
print(f"  num_workers=0（串行）: {serial_ms:7.1f} ms  ← 理论下限 ≈ {n_samples*2:.0f} ms")
print()
print("理论上并行加速（CPU 核足够时）:")
print("  num_workers=0  →  1x   （串行）")
print("  num_workers=2  → ~2x   （两个进程同时取样本）")
print("  num_workers=4  → ~4x   （接近 1/4 耗时）")
print()
print("⚠️ 但有两大坑，这就是为什么不能无脑调大：")
print("  ① 小数据（内存 tensor）：开进程 + pickle 的开销 >> 加载收益 → 反而更慢")
print("  ② Jupyter / 被 fork 的环境：worker 经常崩（spawn 找不到 __main__ 里的类）")
print("     真实工程里先 num_workers=0 跑通，再在 .py 脚本里调大验证")

num_workers — 串行 vs 并行加载的概念演示


数据集 256 样本, batch_size=32, 每个 __getitem__ 模拟 2ms
  num_workers=0（串行）:   759.6 ms  ← 理论下限 ≈ 512 ms

理论上并行加速（CPU 核足够时）:
  num_workers=0  →  1x   （串行）
  num_workers=2  → ~2x   （两个进程同时取样本）
  num_workers=4  → ~4x   （接近 1/4 耗时）

⚠️ 但有两大坑，这就是为什么不能无脑调大：
  ① 小数据（内存 tensor）：开进程 + pickle 的开销 >> 加载收益 → 反而更慢
  ② Jupyter / 被 fork 的环境：worker 经常崩（spawn 找不到 __main__ 里的类）
     真实工程里先 num_workers=0 跑通，再在 .py 脚本里调大验证


## 12. 常用操作速查 — 怎么看 DataLoader 里有什么

```python
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# 看一个 batch 长啥样（不跑完整个 epoch）
x, y = next(iter(loader))

# 遍历整个 epoch
for x, y in loader:
    ...

# 总 batch 数（注意：是 batch 数，不是样本数）
len(loader)

# 样本总数要看 dataset
len(loader.dataset)
```

In [13]:
# --- 常用操作 ---
print("=" * 60)
print("DataLoader 常用操作速查")
print("=" * 60)

dataset = TensorDataset(torch.arange(20).float().unsqueeze(1), torch.arange(20))
loader = DataLoader(dataset, batch_size=6, shuffle=True)

print(f"len(loader)          = {len(loader)}        ← batch 数")
print(f"len(loader.dataset)  = {len(loader.dataset)}       ← 样本数")
print()

# next(iter(loader)) — 取第一个 batch
first_batch = next(iter(loader))
x0, y0 = first_batch
print(f"next(iter(loader)):")
print(f"  x.shape = {tuple(x0.shape)}")
print(f"  y       = {y0.tolist()}")
print()

# 遍历整个 loader
total = 0
for x, y in loader:
    total += len(y)
print(f"遍历完所有 batch，累计样本数 = {total}")
print(f"→ 和 len(dataset) 一致: {total == len(dataset)} ✅")

DataLoader 常用操作速查
len(loader)          = 4        ← batch 数
len(loader.dataset)  = 20       ← 样本数

next(iter(loader)):
  x.shape = (6, 1)
  y       = [4, 12, 14, 17, 7, 5]

遍历完所有 batch，累计样本数 = 20
→ 和 len(dataset) 一致: True ✅


## 📝 核心速查

### Dataset — 定义「怎么取一个样本」

| 方式 | 适用场景 |
|------|---------|
| `TensorDataset(X, y)` | 数据已经在内存里（numpy / tensor） |
| `class MyDataset(Dataset)` | 自定义逻辑，实现 `__len__` + `__getitem__` |
| `random_split(ds, [n1, n2])` | 切分 train / val |
| `torchvision.datasets.*` | 内置图片数据集（CIFAR、MNIST…） |
| `datasets.ImageFolder(root)` | 按文件夹自动分类的图片集 |

### DataLoader 关键参数

| 参数 | 默认 | 建议 |
|------|------|------|
| `batch_size` | `1` | 32 / 64 / 128，看显存 |
| `shuffle` | `False` | 训练 `True`，验证 `False` |
| `num_workers` | `0` | 真实训练 4~8，调试 0 |
| `drop_last` | `False` | 训练时 `True`（BatchNorm 需要稳定 batch） |
| `pin_memory` | `False` | 有 GPU 时 `True` |
| `collate_fn` | 默认 stack | 变长序列要自定义 |

### 常见误区

| ❌ 错误 | ✅ 正确 |
|---------|---------|
| 在 `__init__` 里把所有数据读进来 | 在 `__getitem__` 里按需读（懒加载） |
| `len(loader)` 当成样本数 | 样本数是 `len(loader.dataset)` |
| 验证集也 `shuffle=True` | 验证 `shuffle=False`，保证可复现 |
| Windows/Jupyter 用大 `num_workers` 崩溃 | 先 `num_workers=0` 跑通再调 |
| 在 `__getitem__` 里写共享全局变量 | 保持线程安全（多进程会调用它） |

### 训练循环模板

```python
for epoch in range(num_epochs):
    model.train()                              # 训练模式
    for x, y in train_loader:                  # ← DataLoader 在这里
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

    model.eval()                               # 评估模式
    with torch.no_grad():
        for x, y in val_loader:
            ...
```

👉 下一步：去 `practice/05_custom_dataset.py` 刷题！